# TFM · Notebook 03 — Evaluación en el conjunto de test

**Objetivo (Fase 5 del plan):** medir, **una sola vez**, el rendimiento de los seis modelos
en los 61 pacientes de test que ningún modelo ha visto.

| Qué se mide | Cómo |
|---|---|
| Discriminación | AUC-ROC con IC 95 % por bootstrap (2000 remuestreos) |
| Comparación entre modelos | Diferencia de AUC pareada con IC 95 % por bootstrap |
| Calibración | Curva de calibración (5 grupos), Brier, media predicha frente a observada |
| Rendimiento clínico | Sensibilidad, especificidad, VPP, VPN y matriz de confusión con dos umbrales |

> ⚠️ **Regla del test:** ejecuta este notebook cuando los modelos estén cerrados. Si después
> cambias algo en el notebook 02 por lo que veas aquí, el test deja de ser independiente.

**En Colab:** `Ejecutar todas` y sube `datos_cleveland_limpios.csv` y `modelos_tfm.joblib`.

## 1. Librerías y carga

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix
from sklearn.calibration import calibration_curve

SEMILLA = 42
os.makedirs("figures", exist_ok=True)

try:
    from google.colab import files
    if not (glob.glob("datos_cleveland_limpios*.csv") and glob.glob("modelos_tfm*.joblib")):
        print("Sube datos_cleveland_limpios.csv y modelos_tfm.joblib:")
        files.upload()
except ImportError:
    pass

df = pd.read_csv(sorted(glob.glob("datos_cleveland_limpios*.csv"))[0])
paquete = joblib.load(sorted(glob.glob("modelos_tfm*.joblib"))[0])
import sklearn
if sklearn.__version__ != paquete["sklearn"]:
    print(f"AVISO: los modelos se guardaron con scikit-learn {paquete['sklearn']} y aquí hay "
          f"{sklearn.__version__}. Si algo falla, vuelve a ejecutar el notebook 02 en este entorno.")
modelos, umbrales, TABLA_DF = paquete["modelos"], paquete["umbrales"], paquete["tabla_df"]

## 2. Reconstruir el conjunto de test (misma partición que el notebook 02)

In [ ]:
X = df.drop(columns=["num", "enfermedad", "restecg"])
y = df["enfermedad"]
X_test, y_test = X.loc[paquete["indices_test"]], y.loc[paquete["indices_test"]].values
print(f"Test: {len(y_test)} pacientes, {y_test.sum()} con enfermedad")

## 3. Probabilidades predichas por cada modelo

In [ ]:
def prob_diamond_forrester(datos):
    decada = ((datos["age"] // 10) - 3).clip(0, 3).astype(int)
    return np.array([TABLA_DF[(int(cp), int(sx))][d] / 100
                     for cp, sx, d in zip(datos["cp"], datos["sex"], decada)])

prob = {"Diamond-Forrester (tabla)": prob_diamond_forrester(X_test)}
for nombre, m in modelos.items():
    prob[nombre] = m["modelo"].predict_proba(X_test[m["variables"]])[:, 1]

ORDEN = ["Diamond-Forrester (tabla)", "Logística DF reentrenada", "Logística consulta",
         "Random Forest consulta", "Logística completa", "Random Forest completo"]
pd.DataFrame(prob)[ORDEN].describe().T[["mean", "min", "max"]].round(3)

## 4. Discriminación: AUC con IC 95 % por bootstrap

El bootstrap remuestrea los 61 pacientes con reemplazo 2000 veces y recalcula el AUC.
El IC 95 % son los percentiles 2,5 y 97,5. Se usan **los mismos remuestreos para todos
los modelos**, lo que permite después comparar modelos de forma pareada.

In [ ]:
rng = np.random.default_rng(SEMILLA)
B = 2000
muestras = []
while len(muestras) < B:
    idx = rng.integers(0, len(y_test), len(y_test))
    if y_test[idx].min() != y_test[idx].max():      # hace falta al menos un caso de cada clase
        muestras.append(idx)

auc_boot = {n: np.array([roc_auc_score(y_test[i], prob[n][i]) for i in muestras]) for n in ORDEN}

def ic(valores):
    return np.percentile(valores, [2.5, 97.5])

tabla_auc = pd.DataFrame({
    "AUC": [roc_auc_score(y_test, prob[n]) for n in ORDEN],
    "IC 95 % inf": [ic(auc_boot[n])[0] for n in ORDEN],
    "IC 95 % sup": [ic(auc_boot[n])[1] for n in ORDEN],
}, index=ORDEN).round(3)
tabla_auc

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
for n in ORDEN:
    fpr, tpr, _ = roc_curve(y_test, prob[n])
    estilo = "--" if "Diamond" in n or "DF" in n else "-"
    ax.plot(fpr, tpr, estilo, label=f"{n} ({tabla_auc.loc[n, 'AUC']:.2f})")
ax.plot([0, 1], [0, 1], ":", color="grey")
ax.set_xlabel("1 − especificidad")
ax.set_ylabel("Sensibilidad")
ax.set_title("Curvas ROC en el conjunto de test (n = 61)")
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("figures/fig_roc_test.png", dpi=300)
plt.show()

## 5. Comparaciones pareadas de AUC

Cada comparación responde a una pregunta del TFM. Si el IC 95 % de la diferencia
**incluye el 0**, no hay evidencia de que un modelo discrimine mejor que el otro.

In [ ]:
COMPARACIONES = [
    ("Logística DF reentrenada", "Diamond-Forrester (tabla)", "¿Reentrenar mejora la tabla clásica?"),
    ("Logística consulta", "Logística DF reentrenada", "¿Aportan PA, colesterol, glucemia y ECG?"),
    ("Logística completa", "Logística consulta", "¿Aportan ergometría, fluoroscopia y talio?"),
    ("Random Forest completo", "Logística completa", "¿Mejora el Random Forest a la logística?"),
    ("Random Forest consulta", "Logística consulta", "Ídem, con variables de consulta"),
    ("Logística completa", "Diamond-Forrester (tabla)", "Modelo final frente a la tabla clásica"),
]
filas = []
for a, b, pregunta in COMPARACIONES:
    dif = auc_boot[a] - auc_boot[b]
    obs = tabla_auc.loc[a, "AUC"] - tabla_auc.loc[b, "AUC"]
    lo, hi = ic(dif)
    filas.append([pregunta, f"{a} − {b}", round(obs, 3), round(lo, 3), round(hi, 3),
                  "No" if lo <= 0 <= hi else "Sí"])
tabla_comp = pd.DataFrame(filas, columns=["Pregunta", "Comparación", "Diferencia AUC",
                                          "IC 95 % inf", "IC 95 % sup", "¿IC excluye 0?"])
tabla_comp

## 6. Calibración

Un modelo está bien calibrado si, entre los pacientes a los que asigna un 30 %,
aproximadamente el 30 % tiene enfermedad. Con 61 pacientes solo caben 5 grupos
(quintiles de probabilidad predicha): interprétala con prudencia.

In [ ]:
tabla_cal = pd.DataFrame({
    "Brier": [brier_score_loss(y_test, prob[n]) for n in ORDEN],
    "Media predicha": [prob[n].mean() for n in ORDEN],
    "Proporción observada": [y_test.mean()] * len(ORDEN),
}, index=ORDEN).round(3)

# Referencia: Brier de un "modelo" que asigna a todos la prevalencia del test.
# Un Brier mayor que este indica que el modelo es peor que no usar ninguno.
brier_referencia = brier_score_loss(y_test, np.full(len(y_test), y_test.mean()))
print(f"Brier de referencia (predecir la prevalencia a todos): {brier_referencia:.3f}")
tabla_cal

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
grupos = [["Diamond-Forrester (tabla)", "Logística DF reentrenada"],
          ["Logística completa", "Random Forest completo"]]
for ax, nombres in zip(axes, grupos):
    for n in nombres:
        obs, pred = calibration_curve(y_test, prob[n], n_bins=5, strategy="quantile")
        ax.plot(pred, obs, "o-", label=n)
    ax.plot([0, 1], [0, 1], ":", color="grey", label="Calibración perfecta")
    ax.set_xlabel("Probabilidad predicha")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Proporción observada con enfermedad")
axes[0].set_title("Comparadores clínicos")
axes[1].set_title("Modelos completos")
plt.tight_layout()
plt.savefig("figures/fig_calibracion_test.png", dpi=300)
plt.show()

## 7. Rendimiento clínico con dos umbrales

- **0,5**: umbral por defecto.
- ***Rule-out***: umbral fijado en el notebook 02 para una sensibilidad ≥ 90 % en el
  entrenamiento. En el test la sensibilidad puede quedar algo por debajo: es lo esperable.

Cada proporción lleva su IC 95 % de Wilson: con 28 enfermos y 33 sanos, los intervalos son anchos
y hay que informarlos siempre.

In [ ]:
def wilson(k, n, z=1.96):
    # IC 95 % de Wilson para una proporción k/n. Funciona bien con muestras pequeñas
    # y con proporciones de 0 o 1 (por ejemplo, 28/28), donde el IC clásico falla.
    if n == 0:
        return ""
    p = k / n
    centro = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    mitad = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return f"{max(0, centro - mitad):.3f}–{min(1, centro + mitad):.3f}"

def metricas(y_real, p, umbral):
    tn, fp, fn, tp = confusion_matrix(y_real, (p >= umbral).astype(int), labels=[0, 1]).ravel()
    div = lambda a, b: a / b if b else np.nan
    return {"Umbral": round(umbral, 3), "VP": tp, "FP": fp, "FN": fn, "VN": tn,
            "Sensibilidad": div(tp, tp + fn), "IC 95 % sens.": wilson(tp, tp + fn),
            "Especificidad": div(tn, tn + fp), "IC 95 % esp.": wilson(tn, tn + fp),
            "VPP": div(tp, tp + fp), "IC 95 % VPP": wilson(tp, tp + fp),
            "VPN": div(tn, tn + fn), "IC 95 % VPN": wilson(tn, tn + fn)}

filas = []
for n in ORDEN:
    for etiqueta, u in [("0,5", 0.5), ("Rule-out", umbrales[n])]:
        filas.append({"Modelo": n, "Criterio": etiqueta, **metricas(y_test, prob[n], u)})
tabla_clin = pd.DataFrame(filas).set_index(["Modelo", "Criterio"])
tabla_clin.round(3)

## 8. Guardar tablas para la memoria

In [ ]:
tabla_auc.to_csv("figures/tabla_auc_test.csv")
tabla_comp.to_csv("figures/tabla_comparaciones_auc.csv", index=False)
tabla_cal.to_csv("figures/tabla_calibracion_test.csv")
tabla_clin.round(3).to_csv("figures/tabla_rendimiento_clinico.csv")
pd.DataFrame(prob).assign(enfermedad=y_test).to_csv("predicciones_test.csv", index=False)
print("Guardadas las tablas en figures/ y predicciones_test.csv")
try:
    from google.colab import files
    import shutil
    shutil.make_archive("resultados_test", "zip", "figures")
    files.download("resultados_test.zip")
except ImportError:
    pass

## 9. Qué llevar a la memoria

- **Resultados:** tabla de AUC con IC, figura ROC, tabla de comparaciones pareadas,
  figura y tabla de calibración, y tabla de rendimiento clínico (al menos para el modelo
  final y los comparadores; el resto, a anexos).
- **Redacción:** informa siempre el IC junto al AUC. Si el IC de una diferencia incluye
  el 0, escribe "no se observaron diferencias" y nunca "el modelo X fue superior".
- **Discusión:** compara la calibración de la tabla de Diamond y Forrester con la de la
  logística reentrenada: es la prueba empírica de que una regla de 1979 no se transporta
  sin más a una población remitida a angiografía.
- **AUC del test frente al de la validación cruzada:** con solo 61 pacientes, el AUC del test
  puede salir bastante más alto (o más bajo) que el de la validación cruzada del notebook 02
  por puro azar de la partición. Informa ambos: el de la CV (242 pacientes, 5 pliegues) es la
  estimación más estable; el del test, la evaluación independiente. Si el del test es mayor,
  no lo vendas como "rendimiento esperado".